In [2]:
from pathlib import Path
import duckdb
import polars as pl

# Mengikuti pola path dari Milestone 1
DATA_PATH = Path("../data/raw/train-00000-of-00001.parquet")
OUTPUT_PATH = Path("../data/processed/dataset_bersih.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan di {DATA_PATH}. "
        "Pastikan file sudah diletakkan di folder data/raw/"
    )

# Menggunakan Polars untuk efisiensi pemrosesan data besar
df = pl.read_parquet(DATA_PATH)
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 1,030,393 baris x 4 kolom


text,label_text,source,split
str,str,str,str
"""warung ini dimiliki oleh pengu…","""positive""","""indonlp/indonlu/smsa""","""train"""
"""mohon ulama lurus dan k212 mmb…","""neutral""","""indonlp/indonlu/smsa""","""train"""
"""lokasi strategis di jalan suma…","""positive""","""indonlp/indonlu/smsa""","""train"""
"""betapa bahagia nya diri ini sa…","""positive""","""indonlp/indonlu/smsa""","""train"""
"""duh . jadi mahasiswa jangan so…","""negative""","""indonlp/indonlu/smsa""","""train"""


In [5]:
# Profilling dengan duckDB
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""text""","""VARCHAR""",""" awalnya pakai ini buat kul…","""ðŸ™‹Hay KK DIANAðŸ™,, BANTU S…",1198053,null,null,null,null,null,1030393,0.00
"""label_text""","""VARCHAR""","""negative""","""positive""",3,null,null,null,null,null,1030393,0.00
"""source""","""VARCHAR""","""FDR Dataset""","""yudhaislamisulistya/jokowi-twe…",12,null,null,null,null,null,1030393,0.00
"""split""","""VARCHAR""","""instagram_comment_cyberbullyin…","""validation""",5,null,null,null,null,null,1030393,0.00


In [6]:
# Deteksi missing value
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary
     

kolom,jumlah_null,persen_null
str,u32,f64


In [11]:
# Penanganan Missing Values 
df_missing_handled = df.with_columns([
    pl.col("text").fill_null(""),
    pl.col("label_text").fill_null("tidak diketahui"),
    pl.col("source").fill_null("tidak diketahui"),
    pl.col("split").fill_null("train"),
])

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Null sebelum: 0
Null sesudah: 0


In [12]:
# Deteksi dan penanganan outlier

df_with_len = df_missing_handled.with_columns(
    pl.col("text").str.len_chars().alias("text_length")
)

KOLOM_NUMERIK = "text_length"


q1 = df_with_len[KOLOM_NUMERIK].quantile(0.25)
q3 = df_with_len[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1

batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}, Batas atas: {batas_atas}")


df_outlier_handled = df_with_len.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

Q1=118.0, Q3=347.0, IQR=229.0
Batas bawah: 0.0, Batas atas: 690.5


In [13]:
# Deduplikasi 

KOLOM_KUNCI = 'text'

total_baris = len(df_outlier_handled)
baris_unik = (
    df_outlier_handled[KOLOM_KUNCI].n_unique()
    if KOLOM_KUNCI in df_outlier_handled.columns
    else total_baris
)

print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')


df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 1,030,393
Baris unik berdasarkan text: 1,028,149
Duplikat: 2,244


In [14]:
# Validasi data dasar
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()

validasi

total_baris,pelanggaran_null,pelanggaran_batas
i64,"decimal[38,0]","decimal[38,0]"
1028149,0,0


In [18]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 1,030,393
Baris sesudah cleaning: 1,028,149
Data bersih tersimpan di: ../data/processed/dataset_bersih.parquet
